# Tumor Architecture Prediction with Trained Logistic Regression

**Author: Fabien Bertillot**  
---

## 1. Goal

Predict colorectal tumor architectural classes (glandular, fragmented, budding, solid) from pre‑computed spatial features using a **trained logistic regression model**.

---

## 2. Dependencies

- **Python 3.9+**  
- Libraries: `numpy`, `pandas`, `scikit‑learn`, `matplotlib`, `seaborn`  
- Files required:  
  - `embedding.json` — U-Net features  
  - `model.joblib` —  `sklearn.linear_model.LogisticRegression`  
  - `scaler.joblib` —  `A StandardScaler used to normalize the input features before classification.`  
  - `encoder.joblib` —  `A LabelEncoder used to encode the target labels for classification` 
---

## 3. Notebook Roadmap

1. **Load Dependencies**  
2. **Load Logistic Regression Model**  
3. **Tile Proba Prediction**  
   - Read `_embedding.json`  
   - Save predictions to `probabilities.h5`  

---

## 1) Load Dependencies

In [1]:
import pandas as pd
import joblib
import numpy as np
import os
import glob
import json
from pathlib import PurePath

## 2) Load Logistic Regression Model

In [2]:
# Define the directory where models are saved
save_dir = r''

# Define filenames with relative paths
model_path = os.path.join(save_dir, 'model.joblib')
scaler_path = os.path.join(save_dir, 'standard_scaler.joblib')
label_encoder_path = os.path.join(save_dir, 'label_encoder.joblib')

# Check if the files exist
if not (os.path.exists(model_path) and os.path.exists(scaler_path) and os.path.exists(label_encoder_path)):
    raise FileNotFoundError("One or more model files are missing. Please ensure all files are saved correctly.")

# Load the Logistic Regression model
clf_loaded = joblib.load(model_path)
print("Logistic Regression model loaded successfully.")

# Load the StandardScaler
scaler_loaded = joblib.load(scaler_path)
print("StandardScaler loaded successfully.")

# Load the LabelEncoder
le_loaded = joblib.load(label_encoder_path)
print("LabelEncoder loaded successfully.")

Logistic Regression model loaded successfully.
StandardScaler loaded successfully.
LabelEncoder loaded successfully.


In [3]:
label_mapping = dict(zip(le_loaded.transform(le_loaded.classes_), le_loaded.classes_))

## 3) Tile Prediction

In [4]:
EMBEDDING_DIR = r''
OUT_DIR = r''

json_paths = glob.glob(os.path.join(EMBEDDING_DIR, '*.json'))

for i, json_path in enumerate(json_paths):

    # Replace 'path_to_file.json' with the actual file path
    with open(json_path, 'r') as file:
        data = json.load(file)

    image_name = PurePath(json_path).name.replace('_embedding.json', '')
    out_path = os.path.join(OUT_DIR, f'{image_name}_probabilities.h5')

    tile_predictions = []

    for key, value in data.items():

        x = data[key]['x']
        y = data[key]['y']
        embedding = data[key]['embedding']

        features = np.array(embedding).reshape(1, -1)
        features_scaled = scaler_loaded.transform(features)

        y_pred_prob = clf_loaded.predict_proba(features_scaled)
        epsilon = 1e-12
        y_pred_prob_adjusted = y_pred_prob + epsilon
        entropy_values = -np.sum(y_pred_prob_adjusted * np.log2(y_pred_prob_adjusted), axis=1)

        prob_dict = {'x': x, 'y': y, 'shannon_entropy':entropy_values[0]}
        for label, prob in zip(clf_loaded.classes_, y_pred_prob[0]):
            prob_dict[f'prob_{label_mapping[label]}'] = prob

        tile_predictions.append(prob_dict)

    df_pred = pd.DataFrame(tile_predictions)
    df_pred.to_hdf(out_path, key='df', mode='w')

    if i % 100 == 0:
        print(f'Processed {i} images over {len(json_paths)}')

Processed 0 images over 2523
Processed 100 images over 2523
Processed 200 images over 2523
Processed 300 images over 2523
Processed 400 images over 2523
Processed 500 images over 2523
Processed 600 images over 2523
Processed 700 images over 2523
Processed 800 images over 2523
Processed 900 images over 2523
Processed 1000 images over 2523
Processed 1100 images over 2523
Processed 1200 images over 2523
Processed 1300 images over 2523
Processed 1400 images over 2523
Processed 1500 images over 2523
Processed 1600 images over 2523
Processed 1700 images over 2523
Processed 1800 images over 2523
Processed 1900 images over 2523
Processed 2000 images over 2523
Processed 2100 images over 2523
Processed 2200 images over 2523
Processed 2300 images over 2523
Processed 2400 images over 2523
Processed 2500 images over 2523
